# VAPT-Env — Real GRPO Post-Training (Unsloth + TRL + Llama 3.2 3B)

End-to-end RL post-training on the live SecurityAuditEnv at https://huggingface.co/spaces/Sayuj63/Vapt-env using TRL's `GRPOTrainer` + Unsloth 4-bit Llama 3.2 3B + LoRA r=16 — exactly the canonical OpenEnv hackathon stack.

**Required Colab Secrets** (or paste at getpass prompt):
- `HF_TOKEN` (write scope) from https://huggingface.co/settings/tokens
- `OPENROUTER_API_KEY` from https://openrouter.ai/keys
- `WANDB_API_KEY` from https://wandb.ai/authorize

Hardware: Colab T4 (free) or A100 (paid). Total ~2-3 hrs.

## 1. Install dependencies

In [ ]:
# Visible install — any failure is loud (no %%capture).
!pip install -q --upgrade pip
!pip install -q unsloth trl peft transformers accelerate bitsandbytes datasets
!pip install -q wandb matplotlib openai python-dotenv huggingface_hub
# Install the VAPT-Env project (gives us SecurityAuditEnv with proper WebSocket session state).
!pip install -q --upgrade --force-reinstall --no-deps git+https://github.com/Sayuj63/vapt-env.git
# Quick verify
import importlib, sys
importlib.invalidate_caches()
try:
    from security_audit_env import SecurityAuditEnv, SecurityAuditAction
    print("OK security_audit_env importable")
except Exception as e:
    print("FAIL importing security_audit_env:", e)
    print("Re-run this cell or check pip output above.")
    raise

## 2. Auth secrets (W&B / HF / OpenRouter)

In [ ]:
import os
from google.colab import userdata

def _get_secret(name, prompt=True):
    try:
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    if prompt:
        import getpass
        return getpass.getpass(name + " (paste, then Enter; input is hidden): ")
    return None

os.environ["HF_TOKEN"]           = _get_secret("HF_TOKEN") or ""
os.environ["OPENROUTER_API_KEY"] = _get_secret("OPENROUTER_API_KEY") or ""
wandb_key                        = _get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"]      = wandb_key or ""
USE_WANDB = bool(wandb_key)

os.environ["ENV_URL"]     = "https://Sayuj63-Vapt-env.hf.space"
os.environ["HF_HUB_REPO"] = "Sayuj63/vapt-env-llama32-3b-grpo"

if USE_WANDB:
    import wandb
    wandb.login(key=os.environ["WANDB_API_KEY"])
    print("OK W&B authenticated")
else:
    print("NOTE: W&B disabled — will log locally only")

from huggingface_hub import login as hf_login
hf_login(token=os.environ["HF_TOKEN"])
print("OK HF Hub authenticated")
print("OK env_url =", os.environ["ENV_URL"])

## 3. Connect to live env (uses canonical WebSocket session client)

In [ ]:
from security_audit_env import SecurityAuditEnv, SecurityAuditAction

env_url = os.environ["ENV_URL"]
with SecurityAuditEnv(base_url=env_url).sync() as env:
    r = env.reset(scenario_id="easy")
    print("OK connected to", env_url)
    print("  steps_remaining:", r.observation.steps_remaining)
    print("  message:", (r.observation.message or "")[:120], "...")

## 4. Pre-training baseline — Llama 3.2 3B via OpenRouter (inline)

In [ ]:
import json, re, textwrap
from openai import OpenAI

SYSTEM_PROMPT = textwrap.dedent('''You are a security auditor. Reply with ONE JSON object only - no prose, no code fences.

Three core actions:
  USE TOOL:   {"action_type":"use_tool","tool_name":"<tool>","arguments":{...}}
  SUBMIT:     {"action_type":"submit_finding","arguments":{"title":"...","host":"<ip>","type":"<vuln>","severity":"Critical|High|Medium|Low","cvss_score":<0-10>,"cwe":"CWE-XX","owasp":"AXX:2021 - ...","endpoint":"<path>","evidence":"<why>","remediation":"<fix>"}}
  REPORT:     {"action_type":"generate_report"}

Tools: network_scan, web_crawl, test_injection, test_xss, test_auth, test_config, test_crypto, check_secrets, vulnerability_scan, service_fingerprint.

Examples:
Tool output: "[CRITICAL] SQL Injection at /api/login (param=username), CWE-89, CVSS 9.8"
Reply: {"action_type":"submit_finding","arguments":{"title":"SQL Injection in /api/login","host":"10.0.1.10","type":"SQL Injection","severity":"Critical","cvss_score":9.8,"cwe":"CWE-89","owasp":"A03:2021 - Injection","endpoint":"/api/login","evidence":"Tool flagged param=username","remediation":"Parameterized queries"}}

Tool output: "Discovered host 10.0.1.10 (web)"
Reply: {"action_type":"use_tool","tool_name":"web_crawl","arguments":{"host":"10.0.1.10"}}

Rules:
- Each scenario has a SMALL FIXED number of vulns (~3 easy, ~6 medium, ~10 hard). Do not exceed it.
- ONE finding per unique (host, vuln_type). No duplicates.
- The moment you have NO new evidence, call generate_report.
- Do NOT repeat list_tools or network_scan once called.''').strip()

def render_observation(obs):
    return "\n".join([
        "phase=" + obs.current_phase,
        "hosts=" + str(obs.discovered_hosts or []),
        "services=" + str(obs.discovered_services or {}),
        "findings_submitted=" + str(obs.findings_submitted),
        "steps_remaining=" + str(obs.steps_remaining),
        "tool_output:\n" + (obs.tool_output or "")[:1200],
    ])

def parse_action(text):
    if not text:
        return SecurityAuditAction(action_type="list_tools")
    m = re.search(r"\{[\s\S]*\}", text)
    if not m:
        return SecurityAuditAction(action_type="list_tools")
    try:
        d = json.loads(m.group(0))
        return SecurityAuditAction(
            action_type=d.get("action_type", "list_tools"),
            tool_name=d.get("tool_name"),
            arguments=d.get("arguments") or {},
        )
    except Exception:
        return SecurityAuditAction(action_type="list_tools")

baseline_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
BASELINE_MODEL = "meta-llama/llama-3.2-3b-instruct"

def baseline_episode(scenario_id, max_steps):
    with SecurityAuditEnv(base_url=env_url).sync() as e:
        r = e.reset(scenario_id=scenario_id)
        obs = r.observation
        last_reward = 0.0
        steps_done = 0
        for step in range(max_steps):
            steps_done = step + 1
            try:
                resp = baseline_client.chat.completions.create(
                    model=BASELINE_MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": render_observation(obs)},
                    ],
                    max_tokens=256,
                    temperature=0.5,
                )
                text = resp.choices[0].message.content or ""
            except Exception:
                text = ""
            action = parse_action(text)
            rs = e.step(action)
            obs = rs.observation
            last_reward = float(rs.reward or 0.0)
            if rs.done:
                break
        return last_reward, steps_done

baseline = {}
for sid, mx in (("easy", 25), ("medium", 35), ("hard", 45)):
    print(">>> baseline " + sid, flush=True)
    s, n = baseline_episode(sid, mx)
    baseline[sid] = s
    print("  " + sid + ": " + format(s, ".4f") + " in " + str(n) + " steps")
baseline["average"] = sum(baseline[k] for k in ("easy","medium","hard")) / 3
with open("baseline_scores.json", "w") as f:
    json.dump(baseline, f, indent=2)
print()
print("BASELINE:", json.dumps(baseline, indent=2))

## 5. Load Llama 3.2 3B + LoRA via Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
FastLanguageModel.for_training(model)
print("OK model + LoRA loaded")
model.print_trainable_parameters()

## 6. Build rollout dataset (~28 prompts)

In [ ]:
from datasets import Dataset

def prefix_act(step_idx, first_host):
    acts = [
        SecurityAuditAction(action_type="list_tools"),
        SecurityAuditAction(action_type="use_tool", tool_name="network_scan", arguments={"target":"10.0.0.0/16"}),
        SecurityAuditAction(action_type="use_tool", tool_name="web_crawl", arguments={"host": first_host}),
        SecurityAuditAction(action_type="use_tool", tool_name="test_injection", arguments={"host": first_host, "endpoint":"/api/login"}),
    ]
    return acts[step_idx % len(acts)]

rollout_rows = []
for scenario_id, n_episodes in (("easy", 3), ("medium", 4)):
    for ep in range(n_episodes):
        with SecurityAuditEnv(base_url=env_url).sync() as e:
            r = e.reset(scenario_id=scenario_id)
            obs = r.observation
            first_host = "10.0.1.10"
            for step in range(8):
                rollout_rows.append({
                    "scenario_id": scenario_id,
                    "step": step,
                    "prompt": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": render_observation(obs)},
                    ],
                })
                rs = e.step(prefix_act(step, first_host))
                obs = rs.observation
                if obs.discovered_hosts: first_host = obs.discovered_hosts[0]
                if rs.done: break

ds = Dataset.from_list(rollout_rows)
print("OK rollout dataset:", len(ds), "prompts")
print("-" * 60)
print(ds[0]["prompt"][1]["content"][:400])

## 7. Reward function (env-driven)

In [ ]:
def reward_fn(completions, prompts=None, scenario_id=None, step=None, **kw):
    rewards = []
    n = len(completions)
    sids  = scenario_id if isinstance(scenario_id, list) else [scenario_id] * n
    steps = step        if isinstance(step, list)        else [step]        * n
    for k, comp in enumerate(completions):
        text = comp[0]["content"] if isinstance(comp, list) else comp
        action = parse_action(text)
        try:
            with SecurityAuditEnv(base_url=env_url).sync() as e:
                r = e.reset(scenario_id=sids[k])
                obs = r.observation
                first_host = "10.0.1.10"
                for s in range(steps[k]):
                    rs = e.step(prefix_act(s, first_host))
                    if rs.observation.discovered_hosts:
                        first_host = rs.observation.discovered_hosts[0]
                rs = e.step(action)
                rewards.append(float(rs.reward or 0.0))
        except Exception:
            rewards.append(-0.1)
    return rewards

# Smoke test
sample = ['{"action_type":"use_tool","tool_name":"network_scan","arguments":{"target":"10.0.0.0/16"}}']
print("OK reward_fn smoke:", reward_fn(sample, scenario_id=["easy"], step=[0]))

## 8. GRPO training

In [ ]:
from trl import GRPOConfig, GRPOTrainer
import torch

# T4 = Turing, no bf16. Auto-detect.
_use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
_use_fp16 = torch.cuda.is_available() and not _use_bf16

if USE_WANDB:
    wandb.init(
        project="vapt-env-grpo",
        name="llama32-3b-easy-medium",
        config={"lora_r":16,"lr":5e-6,"num_generations":4,"dataset_size":len(ds)},
    )

cfg = GRPOConfig(
    output_dir="vapt_grpo_out",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    logging_steps=1,
    num_generations=4,
    max_prompt_length=1500,
    max_completion_length=256,
    temperature=0.7,
    save_strategy="no",
    report_to="wandb" if USE_WANDB else "none",
    bf16=_use_bf16,
    fp16=_use_fp16,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
)
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=cfg,
    train_dataset=ds,
)
trainer.train()
wandb_url = wandb.run.url if (USE_WANDB and wandb.run) else "n/a"
if USE_WANDB: wandb.finish()
print("OK training complete. W&B run:", wandb_url)

## 9. Save + push trained LoRA adapter to HF Hub

In [ ]:
model.save_pretrained("vapt_grpo_adapter")
tokenizer.save_pretrained("vapt_grpo_adapter")

from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(os.environ["HF_HUB_REPO"], exist_ok=True, repo_type="model")
api.upload_folder(
    folder_path="vapt_grpo_adapter",
    repo_id=os.environ["HF_HUB_REPO"],
    commit_message="GRPO post-training on VAPT-Env (live HF Spaces env)",
)
print("OK adapter pushed: https://huggingface.co/" + os.environ["HF_HUB_REPO"])

## 10. Post-training eval (with force-report)

GRPO over-optimises submit_finding. Force generate_report after submit_count hits the vuln budget so the grader actually returns a final_score.

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

VULN_BUDGET = {"easy": 3, "medium": 6, "hard": 10}

def run_episode(scenario_id, max_steps):
    submit_count = 0
    target = VULN_BUDGET[scenario_id]
    with SecurityAuditEnv(base_url=env_url).sync() as e:
        r = e.reset(scenario_id=scenario_id)
        obs = r.observation
        last_reward = 0.0
        steps_done = 0
        for step in range(max_steps):
            steps_done = step + 1
            if submit_count >= target:
                action = SecurityAuditAction(action_type="generate_report")
                rs = e.step(action)
                obs = rs.observation
                last_reward = float(rs.reward or 0.0)
                print("  [" + scenario_id + " s" + str(steps_done) + "] FORCED report r=" + format(last_reward, "+.3f"), flush=True)
                break
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": render_observation(obs)},
            ]
            input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
            out = model.generate(
                input_ids, max_new_tokens=256,
                do_sample=True, temperature=0.7, top_p=0.9,
                repetition_penalty=1.3,
                pad_token_id=tokenizer.eos_token_id,
            )
            text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
            action = parse_action(text)
            if action.action_type == "submit_finding":
                submit_count += 1
            rs = e.step(action)
            obs = rs.observation
            last_reward = float(rs.reward or 0.0)
            line = "  [" + scenario_id + " s" + str(steps_done) + "] " + action.action_type
            if action.tool_name: line += "(" + action.tool_name + ")"
            line += " sub=" + str(submit_count) + " r=" + format(last_reward, "+.3f")
            print(line, flush=True)
            if rs.done: break
        return last_reward, steps_done

trained = {}
for sid, mx in (("easy", 25), ("medium", 35), ("hard", 45)):
    print("\n>>> trained " + sid, flush=True)
    score, steps = run_episode(sid, mx)
    trained[sid] = score
    print("  RESULT " + sid + ": " + format(score, ".4f") + " in " + str(steps) + " steps", flush=True)
trained["average"] = sum(trained[k] for k in ("easy","medium","hard")) / 3
with open("trained_scores.json", "w") as f:
    json.dump(trained, f, indent=2)
print()
print("TRAINED:", json.dumps(trained, indent=2))

## 11. Plots — performance comparison + reward / loss curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import json as _json

with open("baseline_scores.json") as f: bsc = _json.load(f)
with open("trained_scores.json")  as f: tsc = _json.load(f)

scenarios = ["easy","medium","hard","average"]
b = [bsc.get(s, 0) for s in scenarios]
t = [tsc.get(s, 0) for s in scenarios]

x = np.arange(len(scenarios))
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - 0.2, b, 0.4, label="Llama 3.2 3B (pre-training)", color="#bbbbbb", edgecolor="black")
ax.bar(x + 0.2, t, 0.4, label="Llama 3.2 3B (post-GRPO)",    color="#2a9d8f", edgecolor="black")
ax.set_xticks(x); ax.set_xticklabels([s.title() for s in scenarios])
ax.set_xlabel("Scenario"); ax.set_ylabel("Final score (0-1)")
ax.set_title("VAPT-Env: Llama 3.2 3B before vs after GRPO post-training")
ax.legend(loc="upper right"); ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0, max(max(b), max(t), 0.1) * 1.25)
for i, (bi, ti) in enumerate(zip(b, t)):
    ax.text(i - 0.2, bi + 0.01, format(bi, ".2f"), ha="center", fontsize=9)
    ax.text(i + 0.2, ti + 0.01, format(ti, ".2f"), ha="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig("performance_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("OK saved performance_comparison.png")

In [ ]:
# W&B reward + loss curves with EXPLICIT TRL/Unsloth column names
import wandb
import matplotlib.pyplot as plt

api = wandb.Api()
runs = api.runs("vapt-env-grpo", order="-created_at")
run = runs[0]
print("W&B run:", run.url)

hist = run.history(keys=["train/reward", "train/loss", "train/global_step",
                         "train/rewards/reward_fn/mean", "train/kl"])
print("rows:", len(hist))

x = hist["train/global_step"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(x, hist["train/reward"], color="#2a9d8f", linewidth=2, label="train/reward")
if "train/rewards/reward_fn/mean" in hist.columns:
    axes[0].plot(x, hist["train/rewards/reward_fn/mean"], color="#264653",
                 linewidth=1.5, alpha=0.6, label="reward_fn/mean")
axes[0].set_xlabel("Training step"); axes[0].set_ylabel("Reward")
axes[0].set_title("GRPO training reward (Llama 3.2 3B + LoRA r=16)")
axes[0].grid(True, alpha=0.3); axes[0].legend()

axes[1].plot(x, hist["train/loss"], color="#e76f51", linewidth=2)
axes[1].set_xlabel("Training step"); axes[1].set_ylabel("Loss")
axes[1].set_title("GRPO loss over training")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("reward_per_episode.png", dpi=150, bbox_inches="tight")
plt.savefig("training_loss.png", dpi=150, bbox_inches="tight")
plt.show()
with open("wandb_run_url.txt", "w") as f: f.write(run.url)
print("OK saved reward_per_episode.png + training_loss.png")

## 12. Final summary + download artifacts

In [ ]:
print("=" * 70)
print("VAPT-Env GRPO post-training summary")
print("=" * 70)
print("Model:    Llama 3.2 3B (Unsloth 4-bit + LoRA r=16)")
print("Env URL:  " + env_url)
print("Adapter:  https://huggingface.co/" + os.environ["HF_HUB_REPO"])
try:
    print("W&B run:  " + open("wandb_run_url.txt").read().strip())
except Exception:
    pass
print()
print(format("",  "<10") + format("baseline", ">10") + format("trained", ">10") + format("delta", ">10"))
for s in ("easy", "medium", "hard", "average"):
    bv = bsc.get(s, 0); tv = tsc.get(s, 0); dv = tv - bv
    print(format(s, "<10") + format(bv, ">10.4f") + format(tv, ">10.4f") + format(dv, ">+10.4f"))
print()
print("Plots: performance_comparison.png, reward_per_episode.png, training_loss.png")

In [ ]:
from google.colab import files
for f in (
    "performance_comparison.png", "reward_per_episode.png", "training_loss.png",
    "baseline_scores.json", "trained_scores.json", "wandb_run_url.txt",
):
    try: files.download(f)
    except Exception as e: print("skip", f, e)